In [1]:
from dask.distributed import Client

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 5
Total threads: 10,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54330,Workers: 5
Dashboard: http://127.0.0.1:8787/status,Total threads: 10
Started: Just now,Total memory: 16.00 GiB
Comm: tcp://127.0.0.1:54368,Total threads: 2
Dashboard: http://127.0.0.1:54370/status,Memory: 3.20 GiB
Nanny: tcp://127.0.0.1:54334,


In [2]:
import os
import gc
import collections.abc
#hyper needs the four following aliases to be done manually.
collections.Iterable = collections.abc.Iterable
collections.Sized = collections.abc.Sized
collections.Sequence = collections.abc.Sequence
collections.Mapping = collections.abc.Mapping
import dask.array as da
from dask_image import ndfilters
import dask_image as di
from dask import delayed
import numpy as np
from pylab import imshow


import numpy as np
#import cupy as cp
from os.path import getsize
import napari
from skimage.io import imread


def mmap_load_chunk(filename, shape, dtype, sl):
    data = np.memmap(filename, mode='r', shape=shape, dtype=dtype)
    result = data[sl]
    del data
    return result
    
def mmap_dask_array(filename, shape, dtype):
    chunks = []
    blocksize=30
    for offset in range(0, shape[0], blocksize):
        chunk = da.from_delayed(
            delayed(mmap_load_chunk)(
                filename,
                shape=shape,
                dtype=dtype,
                sl=slice(offset, min(offset+blocksize, shape[0]))
            ),
            shape=(min(offset+blocksize, shape[0]) - offset, ) + shape[1:],
            dtype=dtype
        )
        chunks.append(chunk)
    return da.concatenate(chunks, axis=0)


filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'
MAXCHUNKSIZE = 1024*288*2*1500000
class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

       # self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = da.empty((1,self.height,self.width),dtype=np.uint16)
        self.daskarray = mmap_dask_array(self.fullpath,(self.nFrames,self.height,self.width),dtype=np.uint16)
        self.app = napari.Viewer()
        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

       # self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = da.empty((1,self.height,self.width),dtype=np.uint16)
        self.daskarray = mmap_dask_array(self.fullpath,(self.nFrames,self.height,self.width),dtype=np.uint16)

        l = self.app.layers['Image']
        l.data = self.array
        


    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = self.nFrames
        totalFrames = end-start
        totalFramesSize = totalFrames*self.frameSize


        stack = self.daskarray[start:end,:,:]#cp.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))
        stack = di.ndfilters.gaussian_filter(stack,sigma=2)
        stack3 = [stack]#[stack.map_blocks(cp.asarray)]
            
        return stack3

    def loadNextNFrames(self,n,compute=True):
        newStacks = self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)
        if self.currentLastFrame == 0:
            self.array = newStacks[0]
        else:
            self.array= da.vstack([self.array, *newStacks])
        del newStacks

        if compute:
            try:
                l = self.app.layers['Image']
                l.data = self.array.compute()
            except:
                self.app.add_image(self.array.compute())
        else:
            try:
                l = self.app.layers['Image']
                l.data = self.array
            except:
                self.app.add_image(self.array)           

        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n,compute=True):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return
        newStack = self.loadWholeStack(start=self.currentLastFrame,end = n)
        
        if self.currentLastFrame == 0:
            self.array = newStacks[0]
        else:
            self.array= da.vstack([self.array, *newStacks])

        if compute:
                
            if self.currentLastFrame == 0:
                    self.app.add_image(self.array.compute())
            else:
                l = self.app.layers['Image']
                l.data = self.array.compute()
        else:
            if self.currentLastFrame == 0:
                self.app.add_image(self.array)
            else:
                l = self.app.layers['Image']
                l.data = self.array
        self.currentLastFrame = self.array.shape[0]



In [3]:

fn = '../sampleImage/'
try:
    tb.r.close()
    del tb.array
    tb.loadFile(fn)
    gc.collect()
except:
    print('nope')
    tb = thorlabsFile(fn)

nope


In [4]:
tb.loadNextNFrames(5000,compute=True)

In [15]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 5
Total threads: 10,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49929,Workers: 5
Dashboard: http://127.0.0.1:8787/status,Total threads: 10
Started: Just now,Total memory: 16.00 GiB
Comm: tcp://127.0.0.1:49966,Total threads: 2
Dashboard: http://127.0.0.1:49975/status,Memory: 3.20 GiB
Nanny: tcp://127.0.0.1:49932,
